## Vaccine Efficacy

In [ ]:
import math
import pymc3 as pm
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

We have the following data from clinical trials of the Pfizer vaccine in 12-15 year olds (https://www.pfizer.com/news/press-release/press-release-detail/pfizer-biontech-announce-positive-topline-results-pivotal).

In [ ]:
n_Pfizer_vaccine = 0
n_Pfizer_placebo = 18

n_Pfizer_total_vaccine = 1131
n_Pfizer_total_placebo = 1129

Pfizer_vaccine_outcomes = np.concatenate([np.zeros(n_Pfizer_total_vaccine - n_Pfizer_vaccine), np.ones(n_Pfizer_vaccine)])
Pfizer_placebo_outcomes = np.concatenate([np.zeros(n_Pfizer_total_placebo - n_Pfizer_placebo), np.ones(n_Pfizer_placebo)])

We can build a probablistic model to describe this noisy measurement process.

In [ ]:
model = pm.Model()

with model:
    
    # Probability of exposure

    p_Pfizer = pm.Beta('p_Pfizer', alpha=1, beta=1)
    
    # Probability of protection

    e_Pfizer = pm.Beta('e_Pfizer', alpha=1, beta=1)
    
    # Likelihood of observed number of positive tests
    
    like_Pfizer_vaccine = pm.Bernoulli('like_Pfizer_vaccine', p=p_Pfizer * (1.0 - e_Pfizer), observed=Pfizer_vaccine_outcomes)
    like_Pfizer_placebo = pm.Bernoulli('like_Pfizer_placebo', p=p_Pfizer, observed=Pfizer_placebo_outcomes)

We run the model with some default settings.

In [ ]:
with model:
    
    trace = pm.sample(2000)

In [ ]:
pm.traceplot(trace)

Finally when we are happy we can generate some final plots and report on the results.

In [ ]:
plt.figure(figsize=(12, 9))

plt.hist(trace['e_Pfizer'], histtype='stepfilled', bins=100, alpha=0.85, color="#467821", density=True)
plt.xlim(0, 1)
plt.title('Pfizer')
plt.xlabel('Probability of protection')

plt.tight_layout()

plt.savefig('results.pdf')